# Kartika Sentinel — Retrain IndoBERT (data campuran)
**Jalankan Runtime → Run all** di Colab (GPU T4).
Label wajib: **FAKTA=0**, **HOAX=1**.
Jangan andalkan Mafindo saja. Notebook ini mengunduh banyak sumber FAKTA (idn-news-az semua outlet + boost TNI/Papua) dan HOAX dari TBH + Mafindo (di-cap ≤35% HOAX) + SEACrowd/BRIN jika terbuka.
Hasil: zip model di Drive + metrik hard test (Yonif 143).


In [ ]:
# 1) Install
!pip -q install datasets transformers accelerate scikit-learn pandas huggingface_hub safetensors
import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')


In [ ]:
# 2) Clone repo (script prepare/finetune/eval)
!git clone --depth 1 https://github.com/Moch-Iqbaal/kartika-sentinel.git /content/kartika-sentinel
%cd /content/kartika-sentinel
!mkdir -p /content/raw /content/out


In [ ]:
# 3) Build mixed dataset (skala penuh — bisa 20–60 menit unduh streaming)
# Cap Mafindo HOAX ~35%. FAKTA dari semua host idn-news-az.
!python training/prepare_data.py \
  --raw /content/raw \
  --out /content/out/data \
  --mafindo-hoax-share 0.35 \
  --max-topic 25000 \
  --max-general 80000 \
  --max-per-host 8000
import pandas as pd
print(pd.read_csv('/content/out/data/composition.csv').to_string(index=False))


In [ ]:
# 4) Fine-tune dari indobenchmark/indobert-base-p1 (atau umn-cyber/indobert-hoax)
BASE = 'indobenchmark/indobert-base-p1'
!python training/finetune.py \
  --train /content/out/data/train.csv \
  --val /content/out/data/val.csv \
  --test /content/out/data/test.csv \
  --base indobenchmark/indobert-base-p1 \
  --tokenizer indobenchmark/indobert-base-p1 \
  --out /content/out/model \
  --epochs 3 \
  --batch 16 \
  --max-len 256


In [ ]:
# 5) Hard test + bandingkan baseline umn-cyber/indobert-hoax
!python training/eval_hard.py \
  --model /content/out/model \
  --tokenizer indobenchmark/indobert-base-p1 \
  --hard /content/out/data/hard_test.csv \
  --baseline umn-cyber/indobert-hoax \
  --baseline-tokenizer indobenchmark/indobert-base-p1 \
  --out /content/out/hard_eval.json
import json
print(json.dumps(json.load(open('/content/out/hard_eval.json')), indent=2)[:2000])


In [ ]:
# 6) Zip + simpan ke Google Drive
from google.colab import drive
drive.mount('/content/drive')
!cp /content/out/data/composition.csv /content/out/model/
!cp /content/out/data/meta.json /content/out/model/
!cp /content/out/hard_eval.json /content/out/model/
!cp /content/out/model/metrics.json /content/out/model/ 2>/dev/null || true
!cd /content/out && zip -r /content/kartika-indobert-hoax-v2.zip model data/hard_test.csv data/composition.csv data/meta.json hard_eval.json
!mkdir -p '/content/drive/MyDrive/KartikaSentinel'
!cp /content/kartika-indobert-hoax-v2.zip '/content/drive/MyDrive/KartikaSentinel/'
print('OK -> Drive/KartikaSentinel/kartika-indobert-hoax-v2.zip')
print('Di Mac: unzip, salin isi model/ ke folder HOAX_MODEL_DIR (default models/umn-cyber-indobert-hoax). JANGAN sentuh .env.onprem.')
